# Diffeomorphic Learning with PyTorch
This notebook implements the exact same dataset evaluation done in the NumPy baseline, but using **PyTorch**.
It also adds the **Affine Transformation ($g$)** referring to translation and rotation as introduced in the presentation's math section (`g(t, x) = A(t) x + b(t)`).
Using PyTorch enables automatic differentiation, so we do not have to manually derive the Hamiltonian PMP backward pass!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles, make_moons, make_blobs, make_gaussian_quantiles
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(42)

In [ ]:
class DiffeomorphicLearnerTorch(nn.Module):
    def __init__(self, n, d, c, T=8, rho=0.2, sigma2=1.0, l2=1e-2):
        super().__init__()
        self.T = T
        self.dt = 1.0 / T
        self.rho = rho
        self.sigma2 = sigma2
        self.l2 = l2
        
        # The velocity control vectors in RKHS
        self.A = nn.Parameter(torch.zeros(T, n, d))
        
        # The Affine transformation components: g(t, x) = M(t) x + b_aff(t)
        # We define a separate matrix for translation and rotation scaling transformation.
        self.A_aff = nn.Parameter(torch.zeros(T, d, d))
        self.b_aff = nn.Parameter(torch.zeros(T, d))
        
        # Final terminal classifier (Logistic Regression) W*x + b
        self.W = nn.Parameter(torch.randn(d, c) * 0.01)
        self.b = nn.Parameter(torch.zeros(c))
        
    def gaussian_kernel(self, X, Y):
        # X: (n1, d), Y: (n2, d)
        XX = (X ** 2).sum(dim=1, keepdim=True)
        YY = (Y ** 2).sum(dim=1, keepdim=True).t()
        dist = XX + YY - 2 * X @ Y.t()
        return torch.exp(-dist / (2 * self.rho ** 2))
        
    def forward_train(self, X):
        Z = X
        Ks = []
        for t in range(self.T):
            K = self.gaussian_kernel(Z, Z)
            Ks.append(K)
            
            # Application of the full velocity field
            # v(t, Z) = g(t, Z) + K @ A[t]
            # g(t, Z) = Z @ A_aff[t]^T + b_aff[t]
            v = Z @ self.A_aff[t].t() + self.b_aff[t].unsqueeze(0) + K @ self.A[t]
            Z = Z + self.dt * v
        return Z, Ks

    def forward_test(self, X_test, X_train):
        Z_train = X_train
        Z_test = X_test
        for t in range(self.T):
            K_train = self.gaussian_kernel(Z_train, Z_train)
            K_cross = self.gaussian_kernel(Z_test, Z_train)
            
            v_train = Z_train @ self.A_aff[t].t() + self.b_aff[t].unsqueeze(0) + K_train @ self.A[t]
            v_test = Z_test @ self.A_aff[t].t() + self.b_aff[t].unsqueeze(0) + K_cross @ self.A[t]
            
            Z_train = Z_train + self.dt * v_train
            Z_test = Z_test + self.dt * v_test
            
        return Z_test

    def get_loss(self, X, y):
        Z_final, Ks = self.forward_train(X)
        
        # Terminal classification loss (Cross Entropy)
        logits = Z_final @ self.W + self.b
        ce_loss = nn.CrossEntropyLoss(reduction='sum')(logits, y) / self.sigma2
        
        # Regularization for terminal classifier
        reg_loss = self.l2 * torch.sum(self.W ** 2)
        
        # RKHS kinetic energy
        # int ||v||^2_V dt = sum_t dt * sum_ij (A_i(t)^T K_ij A_j(t))
        deform_rkhs = 0.0
        for t in range(self.T):
            deform_rkhs += self.dt * torch.sum((Ks[t] @ self.A[t]) * self.A[t])
            
        # Affine kinetic energy
        # int ||g(t)||_{aff}^2 dt = sum_t dt * (0.5 * trace(A_aff A_aff^T) + |b_aff|^2)
        deform_affine = 0.0
        for t in range(self.T):
            deform_affine += self.dt * (0.5 * torch.sum(self.A_aff[t] ** 2) + torch.sum(self.b_aff[t] ** 2))
            
        total_loss = ce_loss + reg_loss + deform_rkhs + deform_affine
        return total_loss


In [ ]:
def train_diffeo_model(X_train, y_train, T=8, rho=0.2, sigma2=1.0, l2=1e-2, lr=2e-2, n_iter=200):
    n, d = X_train.shape
    c = int(y_train.max() + 1)
    
    # Send everything to tensors
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.long)
    
    model = DiffeomorphicLearnerTorch(n=n, d=d, c=c, T=T, rho=rho, sigma2=sigma2, l2=l2)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for it in range(n_iter):
        optimizer.zero_grad()
        loss = model.get_loss(X_t, y_t)
        loss.backward()
        optimizer.step()
        
    # Return the trained model and the original training tensor (as it acts as landmarks for testing)
    return model, X_t

def predict_diffeo_model(model, X_train_t, X_test):
    model.eval()
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        Z_final = model.forward_test(X_test_t, X_train_t)
        logits = Z_final @ model.W + model.b
        preds = torch.argmax(logits, dim=1)
    return preds.numpy()


In [ ]:
def build_dataset(name, seed, n_samples=200):
    if name == "circles":
        X, y = make_circles(n_samples=n_samples, noise=0.05, factor=0.5, random_state=seed)
    elif name == "moons":
        X, y = make_moons(n_samples=n_samples, noise=0.1, random_state=seed)
    elif name == "blobs":
        X, y = make_blobs(n_samples=n_samples, centers=2, n_features=2, random_state=seed)
    elif name == "gaussian_quantiles":
        X, y = make_gaussian_quantiles(n_samples=n_samples, n_features=2, n_classes=2, random_state=seed)
    else:
        raise ValueError(f"Unknown dataset: {name}")
    return X.astype(float), y.astype(int)

def add_one_dimension_with_noise(X2, std=0.01, seed=0):
    rng = np.random.default_rng(seed)
    z = std * rng.normal(size=(len(X2), 1))
    return np.hstack([X2, z])

def estimate_rho_like_paper(X, y):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    same_vals, other_vals = [], []
    for i in range(len(X)):
        same = D[i, y == y[i]]
        same = same[same > 0]
        if len(same): same_vals.append(np.percentile(same, 5))
        other = D[i, y != y[i]]
        if len(other): other_vals.append(np.min(other))
    
    rho1 = np.percentile(same_vals, 75)
    rho2 = np.percentile(other_vals, 10)
    return max(1e-3, min(rho1, rho2))


In [ ]:
DATASET_NAMES = ["circles", "moons", "blobs", "gaussian_quantiles"]
SEEDS = [0, 1, 2]
NOISE_STD_EXTRA_DIM = 0.01
N_SAMPLES = 200
TEST_SIZE = 0.30

# Training constants
T_STEPS = 8
N_ITER = 200
LR_DIFFEO = 2e-2
SIGMA2 = 1.0
L2_WEIGHT = 1e-2

rows = []

for ds_name in DATASET_NAMES:
    for seed in SEEDS:
        X2, y = build_dataset(ds_name, seed=seed, n_samples=N_SAMPLES)
        
        idx = np.arange(len(X2))
        idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE, random_state=seed, stratify=y)
        
        X3 = add_one_dimension_with_noise(X2, std=NOISE_STD_EXTRA_DIM, seed=seed + 1234)
        
        for dim, X in [(2, X2), (3, X3)]:
            Xtr, Xte = X[idx_tr], X[idx_te]
            ytr, yte = y[idx_tr], y[idx_te]
            rho = estimate_rho_like_paper(Xtr, ytr)
            
            # Using our PyTorch implementation with Affine Transforms
            model, Xtr_t = train_diffeo_model(
                Xtr, ytr, 
                T=T_STEPS, rho=rho, sigma2=SIGMA2, l2=L2_WEIGHT, 
                lr=LR_DIFFEO, n_iter=N_ITER
            )
            
            preds = predict_diffeo_model(model, Xtr_t, Xte)
            acc = float((preds == yte).mean())
            
            rows.append({
                "dataset": ds_name,
                "seed": seed,
                "dim": dim,
                "acc_diffeo_torch": acc,
                "rho": rho,
            })
            print(f"Dataset: {ds_name:<18} | Seed: {seed} | Dim: {dim} | Acc (Torch+Affine): {acc:.3f} | Rho: {rho:.3f}")

results_df = pd.DataFrame(rows)


In [ ]:
print("--- SUMMARY (PYTORCH + AFFINE) ---")
summary_df = (
    results_df
    .groupby(["dataset", "dim"], as_index=False)
    .agg(
        acc_mean=("acc_diffeo_torch", "mean"),
        acc_std=("acc_diffeo_torch", "std"),
    )
)
print(summary_df)
